<a href="https://colab.research.google.com/github/nilnil47/simple-committee-machine/blob/main/simple_commette_machine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Minimal Teacher-Student Model (Knowledge Distillation) Implementation

Knowledge Distillation involves training a smaller 'Student' model to mimic the behavior of a pre-trained, larger 'Teacher' model.

### Teacher: Hermite Polynomial $He_3(w_* \cdot x)$

In this setup, the teacher is not a neural network but a fixed direction $w_*$. The target output is the third Hermite polynomial $He_3(z) = z^3 - 3z$ applied to the projection of $x$ onto $w_*$.

In [1]:
import sys
!{sys.executable} -m pip install wandb -q

In [4]:
import wandb

# Ensure parameters are defined for the config
dimension = 10
n_hidden = 32

# Initialize a new W&B run
wandb.init(
    project="hermite-distillation",
    config={
        "learning_rate": 0.001,
        "architecture": "CommitteeStudent",
        "dataset": "Standard Normal Static",
        "epochs": 100,
        "dimension": dimension,
        "n_hidden": n_hidden
    }
)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: neildotan (neildotan-hebrew-university-of-jerusalem) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [11]:
import torch
import torch.nn as nn
import torch.optim as optim
import math
import matplotlib.pyplot as plt

# Parameters
dimension = 10
batch_size = 256
n_hidden = 32

# 1. Define the Teacher
w_star = torch.randn(dimension, 1)
w_star = w_star / torch.norm(w_star)

def hermite_teacher(x, w):
    z = torch.matmul(x, w)
    return (z**3 - 3*z).squeeze()

# 2. Define the Committee Student Model
class CommitteeStudent(nn.Module):
    def __init__(self, d: int, n_hidden: int) -> None:
        super().__init__()
        self.readout_scale = 1.0 / math.sqrt(n_hidden)
        self.W = nn.Parameter(torch.empty(n_hidden, d))
        # Initialize and then normalize to unit Frobenius norm
        nn.init.normal_(self.W, mean=0.0, std=1.0 / math.sqrt(d))
        with torch.no_grad():
            current_norm = torch.norm(self.W)
            self.W.data = self.W.data / current_norm

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = torch.erf(x @ self.W.T)
        return self.readout_scale * h.sum(dim=-1)

student_hermite = CommitteeStudent(dimension, n_hidden)
optimizer = optim.Adam(student_hermite.parameters(), lr=0.001)
criterion = nn.MSELoss()

Setup complete: Model initialized and dataset with 100 samples created.


In [13]:
# 3. Updated Training Loop with Projection and Weight Norm Tracking
train_epochs = 100
smooth_loss = []
weight_norms = []
avg_projections = []
running_loss = 0.0

print(f"Starting training on predefined dataset...")
for epoch in range(train_epochs):
    epoch_loss = 0.0
    epoch_z_sum = 0.0
    num_points = 0

    for x_batch, y_batch in train_loader:
        # Calculate z = x · w_*
        with torch.no_grad():
            z = torch.matmul(x_batch, w_star).squeeze()
            epoch_z_sum += z.mean().item()

        y_pred = student_hermite(x_batch)
        loss = criterion(y_pred, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        num_points += 1

    # Track metrics
    with torch.no_grad():
        current_norm = torch.norm(student_hermite.W).item()
        weight_norms.append(current_norm)

    avg_projections.append(epoch_z_sum / num_points)
    avg_epoch_loss = epoch_loss / len(train_loader)

    if epoch == 0:
        running_loss = avg_epoch_loss
    else:
        running_loss = 0.9 * running_loss + 0.1 * avg_epoch_loss

    smooth_loss.append(running_loss)

    if (epoch + 1) % 5000 == 0:
        print(f"Epoch [{epoch+1}/{train_epochs}], Loss: {running_loss:.4f}, Weight Norm: {current_norm:.4f}")

# Visualize everything including projections
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(rows=2, cols=1,
                    subplot_titles=("Loss and Weight Norm Evolution", "Average Projection (z = x · w_*)"),
                    specs=[[{"secondary_y": True}], [{}]]
)

# Subplot 1: Loss & Norm
fig.add_trace(go.Scatter(y=smooth_loss, name="Loss", line=dict(color="red")), row=1, col=1, secondary_y=False)
fig.add_trace(go.Scatter(y=weight_norms, name="Weight Norm", line=dict(color="blue")), row=1, col=1, secondary_y=True)

# Subplot 2: Projections
fig.add_trace(go.Scatter(y=avg_projections, name="Avg Projection", line=dict(color="green")), row=2, col=1)

fig.update_layout(height=800, title_text="Detailed Training Dynamics")
fig.update_yaxes(title_text="Loss (MSE)", type="log", secondary_y=False, row=1, col=1)
fig.update_yaxes(title_text="Weight Norm", secondary_y=True, row=1, col=1)
fig.update_yaxes(title_text="Value of z", row=2, col=1)
fig.show()

Starting training on predefined dataset...


In [16]:
# 3.1 Training Loop with W&B Logging
import torch
from torch.utils.data import DataLoader, TensorDataset

print(f"Starting training with W&B tracking...")

# 1. Re-ensure the static dataset exists
num_samples = 100
x_data_static = torch.randn(num_samples, dimension)
with torch.no_grad():
    y_data_static = hermite_teacher(x_data_static, w_star)
dataset = TensorDataset(x_data_static, y_data_static)

# 2. Setup DataLoader
train_loader = DataLoader(dataset, batch_size=256, shuffle=True)
train_epochs = 10000

for epoch in range(train_epochs):
    epoch_loss = 0.0
    epoch_z_sum = 0.0
    num_points = 0

    for x_batch, y_batch in train_loader:
        with torch.no_grad():
            # Project x onto w_star
            z = torch.matmul(x_batch, w_star).squeeze()
            epoch_z_sum += z.mean().item()

        y_pred = student_hermite(x_batch)
        loss = criterion(y_pred, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        num_points += 1

    # Calculate metrics for the epoch
    avg_epoch_loss = epoch_loss / len(train_loader)
    avg_z = epoch_z_sum / num_points
    with torch.no_grad():
        current_norm = torch.norm(student_hermite.W).item()

    # Log metrics to W&B
    wandb.log({
        "epoch": epoch,
        "loss": avg_epoch_loss,
        "weight_norm": current_norm,
        "avg_projection_z": avg_z
    })

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{train_epochs}] logged to W&B. Loss: {avg_epoch_loss:.4f}")

# Close the W&B run
wandb.finish()

Starting training with W&B tracking...


Error: You must call wandb.init() before wandb.log()

### Logging Artifacts and Plots to W&B
We can store the ground truth vector `w_star` and model metadata in the run configuration. We can also log the interactive Plotly figure created earlier.

In [15]:
import wandb

# 1. Update the existing run configuration with new metadata
# We use wandb.config.update because the run was already started in cell d13da851
wandb.config.update({
    "w_star": w_star.flatten().tolist(),
    "batch_size": batch_size,
    "activation": "erf"
})

# 2. Log the interactive Plotly figure created in the local training cell
# This allows you to view the interactive version in the W&B dashboard
if 'fig' in globals():
    wandb.log({"training_dynamics_plot": fig})

print("Successfully updated configuration and logged the final plot to W&B.")

# 3. Finalize the run
wandb.finish()

Successfully logged w_star, metadata, and the loss plot to W&B.


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